In [3]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.remote.webelement import WebElement

### We want to scrape BU's classroom page

This enables us to get the building code, address and key attributes such as A/C for every BU classroom 

In [1]:
# NOTE: unnecessary try and excepts are added for debugging purposes.

def get_from_xpath(wait, xpath):
    ''' Helper method for accessing elements on a page with selenium
        The try, excepts are primarily used for debugging purposes
    '''
    try:
        return wait.until(EC.visibility_of_element_located((By.XPATH, xpath)))
    except Exception as e:
        raise e

def get_from_selector(wait, selector):
    ''' Helper method for accessing elements on a page with selenium
        The try, excepts are primarily used for debugging purposes
    '''
    try:
        return wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, selector)))
    except Exception as e:
        raise e

In [4]:
def results_loaded(wait) -> bool:
    ''' Waits for the page to load and checks whether the page errors (once all the classrooms have been queried)
    '''
    try:
        get_from_selector(wait, ".cts-results-list")
        return True
    except Exception as E:
        return False

def classroom_page_loaded(wait) -> bool:
    ''' Waits for a classroom page to load. Important before scraping the page data
    '''
    try:
        get_from_selector(wait, ".cts-detail-container")
        return True
    except Exception as E:
        return False

def get_classroom_links(driver: webdriver.Firefox):
    '''  Start at classroom list page
    '''
    # Get all classroom li elements
    classrooms = driver.find_elements(By.CSS_SELECTOR, ".cts-results-list > li")

    # Return links
    links = []
    for classroom in classrooms:
        atag = classroom.find_element(By.TAG_NAME, "a")
        link = atag.get_attribute("href")
        links.append(link)

    return links

    
def get_classroom_data(driver: webdriver.Firefox):
    ''' Once the driver is redirected to a driver page, it will then read all the data and
        return a dictionary
    '''
    data = {}
    list_elems = []
    list_elems += driver.find_elements(By.CSS_SELECTOR, ".cts-row-alt")
    list_elems += driver.find_elements(By.CSS_SELECTOR, ".cts-alt")
    for list_elem in list_elems:
        spans = list_elem.find_elements(By.TAG_NAME, "span")

        if "cts-icon" in spans[0].get_attribute("class"):
            key = list_elem.text.split('\n')[-1]
            data[key] = True
        else:
            key, value = list_elem.text.split('\n')
            data[key] = value

    return data
        


NOTE: I already ran this block, however, it takes about 40 minutes to complete. I saved all the data in a csv file however.

In [5]:
def scrape(driver):
    ''' Logic for scrapping classrooms
    '''
    wait = WebDriverWait(driver, 4)

    page_ind = 0
    search_link = "https://www.bu.edu/classrooms/find-a-classroom/page/1/?cts_address&cts_capacity_lfa&cts_capacity&cts_filter_submit=Search"

    driver.get(search_link)
    # Check to see if all the classes has been searched
    has_results = results_loaded(wait)

    room_data = []

    while has_results:
        # For each page, go to the classroom link and scrape data from each individual page
        links = get_classroom_links(driver)

        for link in links:
            # This will redirect the browser to the corresponding classroom page
            driver.get(link)
            data = get_classroom_data(driver)
            room_data.append(data)

        # Advance to next page
        page_ind += 1
        search_link = f"https://www.bu.edu/classrooms/find-a-classroom/page/{page_ind}/?cts_address&cts_capacity_lfa&cts_capacity&cts_filter_submit=Search"
        driver.get(search_link)

        # Check to see if all the classes has been searched
        has_results = results_loaded(wait)

    return room_data



In [72]:
options = webdriver.FirefoxOptions()
driver = webdriver.Firefox(options=options)

# Kick off the script
room_data = scrape(driver)

driver.quit()

Save results

In [70]:
import pandas as pd

df = pd.DataFrame(room_data)
df.to_csv("data/classrooms.csv")